# Projeto 1 - T319 (2S2026)

### Instruções

1. Quando você terminar os exercícios do projeto, vá até o menu do Jupyter ou Colab e selecione a opção para fazer download do notebook.
   * Os notebooks têm extensão `.ipynb`.
   * Este deve ser o arquivo que você irá entregar.
   * No Colab, vá até **File → Download → Download .ipynb**.
2. Após o download do notebook, vá até a aba de tarefas do MS Teams, localize a tarefa referente a este projeto e faça o upload do notebook.
3. Atente-se ao prazo de entrega definido na tarefa do MS Teams. Entregas fora do prazo não serão aceitas.
4. **O projeto pode ser resolvido em grupos de, no máximo, 3 alunos.**
5. Todas as questões têm o mesmo peso.
6. Não se esqueça de colocar o(s) nome(s) e número(s) de matrícula no campo abaixo.
7. Você pode consultar todo o material de aula.
8. A interpretação faz parte do projeto. Leia atentamente o enunciado de cada questão e **justifique as respostas discursivas**.
9. Use as sementes pseudoaleatórias indicadas no notebook para que os resultados sejam reproduzíveis.
10. Boa sorte!

---

**Material de apoio da disciplina:** [repositório T319 — Introdução ao Aprendizado de Máquina](https://github.com/zz4fap/t319_aprendizado_de_maquina)


**Nomes e matrículas:**

1.
2.
3.


## Exercícios


# 1) Exercício sobre a escolha do passo de aprendizagem

#### Modelagem da perda de percurso em um sistema de comunicação sem fio

Considere, para fins didáticos, o modelo log-distância com sombreamento

$$
PL(d)=PL(d_0)+10n\log_{10}\left(\frac{d}{d_0}\right)+X_\sigma,
$$

onde:

* $PL(d)$ é a perda de percurso, em dB, para uma distância $d$;
* $PL(d_0)$ é a perda de percurso na distância de referência $d_0$;
* $n$ é o expoente de perda de percurso;
* $X_\sigma\sim\mathcal{N}(0,\sigma^2)$ (variável aleatória Gaussiana normal: média zero e variância dada por $\sigma^2$) representa o sombreamento em dB. **OBS**.: Encare o sombreamento como o ruído que contamina as amostras da perda de percurso, $PL$.

Definindo

$$
x=\log_{10}\left(\frac{d}{d_0}\right),
$$

o valor médio da perda de percurso pode ser escrito como uma reta

$$
y=a_0+a_1x,
$$

com os pesos

$$
a_0=PL(d_0),\qquad a_1=10n.
$$

Neste exercício você irá estimar os pesos $a_0$ e $a_1$ usando o algoritmo do **gradiente descendente em mini-lotes** e comparar a solução com a obtida pela classe `LinearRegression`, a qual implementa a forma fechada, i.e., a equação normal.


**1.** Execute a célula abaixo para importar as bibliotecas e definir a função de gradiente descendente em mini-lotes.

**DICAS**

* Em cada iteração, o gradiente é calculado utilizando um subconjunto de amostras de treinamento; por isso, esta versão é chamada de gradiente descendente em mini-lotes (*mini-batch gradient descent*).
* O passo de aprendizagem varia segundo

$$
\alpha(t)=\frac{\alpha_0}{1+kt},
$$

onde $\alpha_0$ é o passo inicial, $k$ é a taxa de decaimento e $t$ é a iteração.
* O código armazena os EQMs **depois** de cada atualização dos pesos, mantendo coerência entre o histórico de pesos e o histórico de erros.
* O mini-lote tem um tamanho de 10 amostras.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import random

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Reseta os geradores pseudo-aleatórios.
seed = 0
np.random.seed(seed)
random.seed(seed)

# Função usada para a redução do passo de aprendizado.
def time_based_decay(alpha_init, k, iteration):
    """Decaimento temporal do passo de aprendizagem."""
    return alpha_init / (1.0 + k * iteration)

# Função que implementa o gradiente descendente em mini-lotes.
def gradient_descent_mini_batch(X_train, y_train, X_val, y_val,
                           alpha_init=0.01, k=0.0, batch=10,
                           num_epochs=3000):
    """
    Gradiente descendente em mini-lotes para regressão linear.

    Entradas
    -------
    X_train : ndarray
        Amostras de treinamento.
    y_train : ndarray
        Valores objetivos de treinamento.
    X_val : ndarray
        Amostras de validação.
    y_val : ndarray
        Valores objetivos de validação.
    alpha_init : float
        Passo de aprendizagem inicial.
    k : float
        Taxa de decaimento do passo de aprendizagem.
    batch : int
        Número de amostras por iteração usadas para a estimativa do vetor gradiente.
    num_epochs : int
        Número máximo de épocas.

    Retorna
    -------
    a : ndarray
        Pesos ao final do treinamento.
    mse_train_history : ndarray
        Histórico do EQM de treinamento.
    mse_val_history : ndarray
        Histórico do EQM de validação.
    weights_history : ndarray
        Histórico dos pesos.
    alpha_history : ndarray
        Histórico do passo de aprendizagem.
    """
    X_train = np.asarray(X_train, dtype=float)
    y_train = np.asarray(y_train, dtype=float).reshape(-1)
    X_val = np.asarray(X_val, dtype=float)
    y_val = np.asarray(y_val, dtype=float).reshape(-1)

    n_train, n_features = X_train.shape

    # Inicialização dos pesos.
    a = np.zeros(n_features, dtype=float)

    mse_train_history = np.zeros(num_epochs*(n_train//batch))
    mse_val_history = np.zeros(num_epochs*(n_train//batch))
    weights_history = np.zeros((num_epochs*(n_train//batch), n_features))
    alpha_history = np.zeros(num_epochs*(n_train//batch))

    # Itera ao longo de todo o conjunto de treinamento.
    for epoch in range(num_epochs):

      # Embaralha os indices do conjunto de treinamento.
      shuffled_indexes = random.sample(range(0, n_train), n_train)

      # Itera ao longo das amostras do mini-lote.
      for iteration in range(n_train//batch):

          # Seleciona um mini-lote.
          start = iteration*batch
          end = batch*(iteration+1)
          batch_idx = shuffled_indexes[start:end]
          X_train_batch = X_train[batch_idx]
          y_train_batch = y_train[batch_idx]

          # Predição com os pesos atuais.
          y_hat = X_train_batch @ a
          error = y_train_batch - y_hat

          # Gradiente do EQM usando todas as amostras de treinamento.
          gradient = -(2.0 / batch) * (X_train_batch.T @ error)

          # Passo de aprendizagem da iteração.
          alpha = time_based_decay(alpha_init, k, epoch*(n_train//batch) + iteration)

          # Atualização dos pesos.
          a = a - alpha * gradient

          # Calcula os erros usando os NOVOS pesos.
          y_train_batch_hat = X_train_batch @ a
          y_val_hat = X_val @ a

          mse_train_history[epoch*(n_train//batch) + iteration] = mean_squared_error(y_train_batch, y_train_batch_hat)
          mse_val_history[epoch*(n_train//batch) + iteration] = mean_squared_error(y_val, y_val_hat)
          weights_history[epoch*(n_train//batch) + iteration] = a
          alpha_history[epoch*(n_train//batch) + iteration] = alpha

    return a, mse_train_history, mse_val_history, weights_history, alpha_history

**2.** Execute a célula abaixo para gerar artificialmente as amostras deste exercício. Considere

$$
PL(d_0)=30\text{ dB},\quad n=3,\quad d_0=1\text{ m},\quad \sigma=4\text{ dB}.
$$

Observe que o regressor recebe como atributo $x=\log_{10}(d/d_0)$, pois é nessa variável que o valor médio do modelo é linear.


In [ ]:
# Parâmetros do problema.
N = 300
PL0 = 30.0
n_real = 3.0
d0 = 1.0
sigma = 4.0

# Distâncias e atributo linearizado.
d = np.linspace(1.0, 1000.0, N)
x = np.log10(d / d0).reshape(-1, 1)

# Parâmetros verdadeiros do modelo linear.
a0_real = PL0
a1_real = 10.0 * n_real

# Função objetivo (sem sombreamento) e observações ruidosas.
y_sem_ruido = a0_real + a1_real * x
w = np.random.normal(loc=0.0, scale=sigma, size=N).reshape(-1, 1)
y = y_sem_ruido + w

print(f"a0 real = {a0_real:.2f}")
print(f"a1 real = {a1_real:.2f}")
print(f"n real  = {n_real:.2f}")

plt.figure(figsize=(9, 5))
plt.scatter(x, y, s=18, alpha=0.55, label="Função observável")
plt.plot(x, y_sem_ruido, linewidth=2.0, label="Função objetivo")
plt.xlabel(r"$x=\log_{10}(d/d_0)$")
plt.ylabel("Perda de percurso (dB)")
plt.title("Dados sintéticos do modelo log-distância")
plt.grid()
plt.legend()
plt.show()


**3.** Sabendo que o ruído adicionado às amostras possui média zero e desvio padrão igual a $4$ dB, qual é aproximadamente o menor EQM **esperado** para um regressor capaz de representar corretamente a função objetivo? **Justifique sua resposta.**


**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>


**4.** Plote as superfícies de erro e de contorno considerando a reta que modela a perda por percurso como a função hipótese.

**DICAS**:
+ Considere o intervalo de variação de $a_0$ entre 30 e 38 e o de $a_1$ entre 24 a 32.

In [ ]:
# Digite aqui o código do exercício.

**5.** Após avaliar as superfícies de erro e de contorno, responda:

+ Elas mostram uma superfície com formato de tigela, ou seja, circular, onde o erro varia de forma similar em ambas as direções?
+ Caso não, o que deve ser feito para deixar a superfície mais circular e melhorar a convergência do gradiente descendente?

**(Justifique suas respostas)**

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

**6.** Divida o conjunto de amostras em treinamento e validação, utilizando:

* 75% das amostras para treinamento;
* 25% das amostras para validação;
* `random_state=seed`.
* Inclua a variável `d`, gerada no item 2, na divisão das amostras. Isso é feito como mostrado abaixo:
```python
X_train, X_val, y_train, y_val, d_train, d_val = train_test_split(
    x, y, d,
    test_size=0.25,
    random_state=seed
)
```


In [ ]:
# Digite aqui o código do exercício.

**7.** Aplique padronização aos dados dos conjuntos de treinamento e validação. Após a padronização, concatene o vetor do atributos de bias (i.e., vetor com valores constantes iguais a 1) a ambos os conjuntos padronizados.

**DICA**:

+ Os parâmetros de padronização (i.e., média e desvio padrão) devem ser estimados com o conjunto de treinamento e aplicados a ambos os subconjuntos.


In [ ]:
# Digite aqui o código do exercício.

**8.** Execute a célula abaixo para treinar o modelo com diferentes valores do passo inicial $\alpha_0$ e da taxa de decaimento $k$. Analise a influência desses parâmetros sobre a velocidade e a estabilidade da convergência.

Os valores testados são:

* $\alpha_0\in\{0.01,0.03,0.05,0.1\}$;
* $k\in\{0.001,0.003,0.01,0.03\}$.

In [ ]:
alphas = [0.01, 0.03, 0.05, 0.1]
ks = [0.001, 0.003, 0.01, 0.03]
num_epochs = 10

fig, axes = plt.subplots(1, len(ks), figsize=(17, 4.8), sharey=True)

summary_rows = []

for ax, k_value in zip(axes, ks):
    for alpha_value in alphas:
        a_gd, mse_train_hist, mse_val_hist, _, _ = gradient_descent_mini_batch(
            X_train, y_train, X_val, y_val,
            alpha_init=alpha_value,
            k=k_value,
            batch=10,
            num_epochs=num_epochs
        )

        ax.plot(mse_train_hist, label=fr"$\alpha_0$={alpha_value}")

        summary_rows.append({
            "alpha": alpha_value,
            "k": k_value,
            "EQM_treinamento_final": mse_train_hist[-1],
            "EQM_validacao_final": mse_val_hist[-1]
        })

    ax.set_title(fr"$k={k_value}$")
    ax.set_xlabel("Iteração")
    ax.grid()
    ax.legend()

axes[0].set_ylabel("EQM de treinamento")
plt.suptitle("Convergência do gradiente descendente em mini-lotes")
plt.tight_layout()
plt.show()

summary_gd = pd.DataFrame(summary_rows)
summary_gd.sort_values("EQM_treinamento_final").head(15)

**9.** Com base nas curvas e na tabela do item anterior, escolha os valores de $\alpha_0$ e $k$ mais adequados. Justifique sua escolha considerando:

* velocidade de convergência;
* estabilidade;
* valor final dos EQMs de treinamento e validação.


**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>


**10.** Utilizando os valores adequados de $\alpha_0$ e $k$ escolhidos no item anterior, treine novamente o modelo usando o gradiente descendente em mini-lotes e apresente (i.e., imprima):

* $\hat a_0$ e $\hat a_1$;
* $\hat n=\hat a_1/10$;
* EQM de treinamento;
* EQM de validação.


In [ ]:
# Digite aqui o código do exercício.

**11.** Treine agora um modelo com `LinearRegression` e apresente as mesmas grandezas do item anterior.

**DICA:**
+ Use `fit_intercept=False`, pois a matrizes de treinamento e validação já possuem uma coluna de valores iguais a 1 para representar o bias.


In [ ]:
# Digite aqui o código do exercício.

**12.** Compare os resultados obtidos com o gradiente descendente com os obtidos pelo `LinearRegression`. Os pesos e os EQMs ficaram próximos? Explique eventuais diferenças.

**(Justifique sua resposta)**


**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

**13.** Apresente, em uma única figura:

* as amostras observadas;
* a função objetivo;
* a função estimada pelo gradiente descendente em mini-lotes;
* a função estimada pelo `LinearRegression`.

**DICA**:
+ Use `d_train` (criado no item 6) e as predições obtidas com `X_train` para plotar as funções estimadas pelo gradiente descendente em mini-lotes e pelo `LinearRegression`.


In [ ]:
# Digite aqui o código do exercício.

**14.** O modelo foi capaz de estimar adequadamente o expoente de perda de percurso usado na geração das amostras? Explique também a influência do sombreamento sobre os resultados. **(Justifique suas respostas)**


**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>


# 2) Exercício sobre regressão polinomial e validação cruzada

Alguns engenheiros realizaram a coleta de amostras referentes à perda de percurso em um sistema de comunicação sem fio e pediram a você, engenheiro de dados, que encontre uma aproximação para os dados coletados de forma que a perda possa ser estimada dada uma distância qualquer.

O objetivo deste exercício é usar regressão polinomial e validação cruzada para selecionar uma ordem que represente bem a relação entre os dados sem introduzir complexidade desnecessária.


**1.** Execute a célula abaixo para baixar as amostras e as visualizar.

**DICAS**

+ O conjunto total de dados já está dividido em conjuntos de treinamento e teste.
+ Os conjuntos de treinamento e teste já se encontram padronizados.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import random

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

# Define a semente a ser usada.
seed = 42

# Reseta gerador de sequências pseudo-aleatórias.
np.random.seed(seed)

# Baixa a base de dados do repositório.
url = "https://raw.githubusercontent.com/zz4fap/t319_aprendizado_de_maquina/main/projeto/amostras_path_loss.csv"
df = pd.read_csv(url)

d = df["d"].to_numpy()   # distâncias (m)
d = d.reshape(len(d), 1)
y = df["y"].to_numpy()   # perda de percurso (dB)

# Divide o conjunto total de amostras em conjuntos de treinamento e validação.
d_train, d_test, y_train, y_test = train_test_split(
    d, y,
    test_size=0.2,
    random_state=seed
)

# Padroniza os conjuntos.
scaler = StandardScaler()

scaler.fit(d_train)
d_train = scaler.transform(d_train)
d_test = scaler.transform(d_test)

# Apresenta as amostras observadas/coletadas.
plt.figure(figsize=(10, 5))
plt.scatter(d, y, s=18, alpha=0.55, label="Função observável")
plt.xlabel("Distância (m)")
plt.ylabel("y (dB)")
plt.title("Amostras coletadas")
plt.grid()
plt.legend()
plt.show()

**2.** Utilize validação cruzada **k-Fold** para determinar a ordem mais adequada do modelo polinomial. Use o conjunto de treinamento com o **k-fold**.

Considere:

* ordens de 1 até 50, inclusive;
* $K=10$;
* `shuffle=True`;
* `random_state=seed`;
* `scoring='neg_mean_squared_error'`.

Para cada ordem, calcule:

* média do EQM de validação;
* desvio padrão do EQM.

Apresente os dois gráficos em função da ordem.

**DICAS**

* Use um `Pipeline` com `PolynomialFeatures`, `StandardScaler` e `LinearRegression`.
* Faça o parâmetro `include_bias=False` do `PolynomialFeatures` para evitar que a coluna do atributo de bias seja zerada pelo padronizador.
* A função `cross_val_score` devolve valores negativos quando `scoring='neg_mean_squared_error'`; multiplique o resultado por $-1$ para obter o EQM.
* Este procedimento segue a mesma metodologia usada nos exemplos apresentados em sala de aula.


In [ ]:
# Digite aqui o código do exercício.

**3.** Observe os gráficos do item anterior e responda, qual ordem deve ser escolhida? Utilizando o princípio da navalha de Occam, indique qual modelo você escolheria e justifique.


**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

**4.** Utilizando a ordem escolhida, treine um modelo com o conjunto de treinamento e calcule os EQMs com os conjuntos de treinamento e de teste.


In [ ]:
# Digite aqui o código do exercício.

**5.** Compare os valores dos erros de treinamento e teste e responda:
+ Podemos dizer que o modelo tem uma boa capacidade de generalização?
**(Justifique usa resposta)**



**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>


**6.** Apresente uma figura contendo as amostras:

* observadas;
* da função estimada pelo modelo escolhido.


In [ ]:
# Digite aqui o código do exercício.

**7.** Treine agora um modelo polinomial de ordem 1 utilizando as amostras de treinamento. Apresente os EQMs de treinamento e de teste e uma figura comparando as amostras observadas (i.e., ruidosas) e a função estimada.


In [ ]:
# Digite aqui o código do exercício.

**8.** Compare o modelo de ordem 1 com o modelo selecionado pela validação cruzada. O comportamento do modelo de ordem 1 caracteriza *underfitting* (subajuste) ou *overfitting* (sobreajuste)? **Justifique.**

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

# 3) Exercício sobre regressão para a predição do preço de casas.

Neste exercício, você irá desenvolver um modelo de **regressão linear** para prever o preço médio de imóveis em distritos da Califórnia.

O problema é clássico em Machine Learning e consiste em estimar o valor de casas com base em características (i.e., atributos) socioeconômicas e geográficas da região.

Esse tipo de aplicação é amplamente utilizado em:

* Avaliação imobiliária automatizada
* Sistemas de recomendação de investimento
* Planejamento urbano
* Análise socioeconômica

O dataset utilizado é o **California Housing**, derivado do censo dos EUA de 1990, contendo **20.640 amostras e 8 atributos numéricos** [1].

Cada amostra representa um **bloco censitário**, ou seja, uma pequena região geográfica com características demográficas homogêneas.

### **Descrição da Base de Dados**

### Atributos (Features)

| Atributo     | Tipo              | Descrição                               |
| ------------ | ----------------- | --------------------------------------- |
| `MedInc`     | Numérico contínuo | Renda mediana dos moradores do distrito |
| `HouseAge`   | Numérico contínuo | Idade mediana das casas                 |
| `AveRooms`   | Numérico contínuo | Número médio de cômodos por residência  |
| `AveBedrms`  | Numérico contínuo | Número médio de quartos                 |
| `Population` | Numérico contínuo | População total do distrito             |
| `AveOccup`   | Numérico contínuo | Número médio de pessoas por residência  |
| `Latitude`   | Numérico contínuo | Latitude geográfica                     |
| `Longitude`  | Numérico contínuo | Longitude geográfica                    |


### Variável alvo

| Nome          | Tipo              | Descrição                                                                            |
| ------------- | ----------------- | ------------------------------------------------------------------------------------ |
| `MedHouseVal` | Numérico contínuo | Valor médio das casas no distrito (em centenas de milhares de dólares) [2] |

Todos os atributos são numéricos e contínuos, o que torna o dataset ideal para regressão [3].


## **Objetivo do Exercício**

Desenvolver um modelo capaz de prever o preço médio das casas utilizando:

* Regressão Linear
* Validação cruzada
* Avaliação com métricas de regressão

### **Referências**:

[1] https://huggingface.co/datasets/gvlassis/california_housing?utm_source=chatgpt.com "gvlassis/california_housing · Datasets at Hugging Face"

[2] https://www.kaggle.com/datasets/shraddha4ever20/california-housing-dataset?utm_source=chatgpt.com "California Housing dataset"

[3] https://sequential-parameter-optimization.github.io/spotPython/reference/spotpython/data/california_housing/?utm_source=chatgpt.com "california_housing - spotpython"


**1.** Execute a célula abaixo para baixar a base de dados. Após a execução, as 10 primeiras linhas da base dedados são mostradas.

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

# Defina uma semente a ser usada ao longo de todo o exercício.
seed = 42

# Carregar dataset
data = fetch_california_housing(as_frame=True)
df = data.frame

# Visualizar dados
df.head(10)

**2.** Execute o comando abaixo para obter uma visão geral da estrutura do conjunto de dados.

O método `df.info()` é utilizado para exibir um resumo conciso do DataFrame, fornecendo informações importantes sobre sua estrutura.

Ao executar esse comando, são apresentadas as seguintes informações:
+ Tipo do objeto (DataFrame do Pandas)
+ Número total de registros (linhas)
+ Intervalo do índice (ex: de 0 a 20639)
+ Quantidade de colunas
+ Nome de cada coluna
+ Número de valores não nulos em cada coluna
+ Tipo de dado (dtype) de cada atributo (ex: float64)
+ Uso de memória do DataFrame

Esse método é extremamente útil na fase inicial de análise exploratória dos dados, pois permite verificar rapidamente:
+ Se existem valores ausentes
+ Se os tipos de dados estão corretos
+ O tamanho do dataset

In [ ]:
# Informações gerais
df.info()

**3.** Com base no resultado do método `df.info()`, responda:

+ O dataset possui valores ausentes? Justifique sua resposta utilizando as informações apresentadas na saída do método.

**Resposta:**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>


**4.** Execute o código abaixo para obter um resumo estatístico das variáveis numéricas do dataset.

O método `df.describe()` gera um conjunto de **estatísticas descritivas** que ajudam a entender a distribuição dos dados em cada coluna numérica do DataFrame.

Ele calcula automaticamente, para cada atributo:

* **count** → número de valores (não nulos)
* **mean** → média
* **std** → desvio padrão (dispersão dos dados)
* **min** → valor mínimo
* **25% (Q1)** → primeiro quartil
* **50% (mediana)** → valor central
* **75% (Q3)** → terceiro quartil
* **max** → valor máximo

Essas informações permitem analisar:

* A **distribuição dos dados**
* A presença de **valores extremos (outliers)**
* A **escala das variáveis**
* A variabilidade entre os atributos

**Interpretação do resultado apresentado**

A saída mostra uma tabela onde cada coluna corresponde a um atributo do dataset, e cada linha representa uma métrica estatística.



In [ ]:
df.describe()

**5.** Com base nos valores apresentados em `df.describe()`, analise o intervalo de variação dos atributos (`min`e `max`), verifique se eles são muito diferentes entre eles e responda se devemos aplicar escalonamento de atributos. (**Justifique sua resposta**)




**Resposta:**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>


**6.** Execute o código abaixo para visualizar a relação entre as variáveis do dataset.

**Correlação**

O método `corr()` calcula a matriz de correlação entre todas as variáveis numéricas do DataFrame, incluindo atributos e rótulo.

Cada valor varia entre -1 e +1:
+ +1 → correlação positiva forte
+ 0 → ausência de relação linear
+ -1 → correlação negativa forte

Ou seja, ele mede o quanto duas variáveis variam juntas.

**Mapa de calor (heatmap)**

O `heatmap` transforma a matriz em um gráfico colorido.
A escala de cores indica a intensidade da correlação:
+ Vermelho → correlação positiva
+ Azul → correlação negativa

**Interpretação do resultado**

A saída é uma matriz simétrica, onde:
+ A diagonal principal sempre será 1 (variável correlacionada consigo mesma)
+ Valores mais próximos de ±1 indicam relações fortes
+ Valores próximos de 0 indicam pouca ou nenhuma relação linear

In [ ]:
# Plota a matriz de correlação
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title("Matriz de Correlação")
plt.show()

**7.** Com base na matriz de correlação, responda:

+ Qual variável apresenta maior influência linear sobre o preço das casas (MedHouseVal)?

Justifique sua resposta utilizando os valores observados no heatmap.

**Resposta:**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>


**8.** Execute o código abaixo. Ele cria seis novos atributos no conjunto de dados e mostra as 5 primeiras linhas do conjunto.

A criação de novos atributos é chamada de engenharia de atributos e pode ajudar a melhorar o desempenho dos modelos de predição. A seguir, apresenta-se uma explicação sobre cada novo atributo criado.

+ **Rooms_per_Household** divide o número médio de cômodos por domicílio (AveRooms) pela média de ocupantes por domicílio (AveOccup). Isso revela quantos cômodos cada pessoa, em média, tem à sua disposição, eliminando o efeito do tamanho do agregado familiar.

+ **Bedrooms_per_Room** representa a proporção entre o número médio de quartos (AveBedrms) e o total de cômodos (AveRooms). Quanto menor essa razão, mais cômodos não destinados a dormitórios (salas, cozinhas, etc.) existem em relação ao número de quartos, o que pode indicar imóveis maiores ou com mais áreas sociais.

+ **Population_per_Household** divide a população total do distrito (`Population`) pelo número médio de ocupantes por domicílio (`AveOccup`). O resultado é uma estimativa do número de domicílios no distrito, permitindo comparar o tamanho populacional independentemente da lotação média das residências.

+ **MedInc_per_Capita** resulta da divisão da renda mediana do distrito (`MedInc`) pelo número médio de ocupantes por domicílio (`AveOccup`). Essa razão aproxima a renda mediana por pessoa, ajustando o poder aquisitivo ao tamanho do agregado familiar e removendo o efeito de famílias numerosas sobre a métrica original de renda domiciliar.

+ **PopDensityIndex** é obtido calculando-se `Population / (AveRooms * AveOccup)`. A expressão `AveRooms * AveOccup` estima a quantidade total de cômodos no distrito, portanto o índice representa quantos habitantes existem, em média, para cada cômodo disponível. Valores altos indicam maior adensamento residencial, sinalizando imóveis mais ocupados e possivelmente menos valorizados.

+ **Rooms_per_Bedroom** é a divisão do número médio de cômodos por domicílio (`AveRooms`) pelo número médio de quartos (`AveBedrms`). Essa proporção revela quantos cômodos (salas, cozinhas, banheiros, etc.) existem para cada quarto no domicílio típico. Quanto maior o valor, mais amplo e diversificado tende a ser o imóvel, sugerindo plantas maiores ou com mais áreas de convivência.

**OBS**.: Observe que no código abaixo nós guardamos uma cópia do dataset sem engenharia de atributos para uso futuro.

In [ ]:
# Guardamos uma cópia do dataset sem engenharia de atributos.
df_no_feature_eng = df.copy()

# Criando as novas colunas
df['Rooms_per_Household'] = df['AveRooms'] / df['AveOccup']
df['Bedrooms_per_Room'] = df['AveBedrms'] / df['AveRooms']
df['Population_per_Household'] = df['Population'] / df['AveOccup']
df['MedInc_per_Capita'] = df['MedInc'] / df['AveOccup']
df['PopDensityIndex'] = df['Population'] / (df['AveRooms'] * df['AveOccup'])
df['Rooms_per_Bedroom']  = df['AveRooms'] / df['AveBedrms']

df.head()

**9.** Execute o código abaixo para visualizar a relação entre as variáveis do dataset. Preste atenção às correlações entre os novos atributos e a variável alvo (`MedHouseVal`).

In [ ]:
# As próximas duas linhas são usadas para manter a variável alvo ('MedHouseVal') como a última coluna do dataset.
col = df.pop('MedHouseVal')
df['MedHouseVal'] = col

# Plota a matriz de correlação
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title("Matriz de Correlação")
plt.show()

**10.** Com base na matriz de correlação acima, responda:

+ Os novos atributos apresentam boa correlação linear com o preço das casas (MedHouseVal)?
+ Quais dos novos atributos têm alta correlação com a variável alvo (MedHouseVal)?
+ Essas correlações são maiores do que as que existiam com os atributos originais.

Justifique sua resposta utilizando os valores observados no heatmap.

**Resposta:**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>


**11.** Execute a célula abaixo para criar a matriz de atributos e o vetor de rótulos.

In [ ]:
# Cria um novo DataFrame removendo a coluna MedHouseVal, ou seja, mantém apenas as variáveis de entrada (X).
# O método drop() é usado para remover a coluna 'MedHouseVal' do DataFrame.
X = df.drop(columns=['MedHouseVal'])

# Seleciona apenas a coluna MedHouseVal, que será a variável alvo (y), ou seja, aquilo que o modelo deve prever
y = df['MedHouseVal']

**12.** Divida o conjunto total de amostras em 80% para treinamento e 20% para teste.

In [ ]:
# Digite o código do exercício aqui.

**13.** Padronize os conjuntos de dados de treinamento e de teste.

In [ ]:
# Digite o código do exercício aqui.

**14.** Instancie e treine um modelo de regressão linear (`LinearRegression`). Apresente o erro quadrático médico (EQM) calculado para os conjuntos de treinamento e teste.

In [ ]:
# Digite o código do exercício aqui.

**15.** Execute a célula abaixo. Ela utiliza o conjunto de dados **sem os novos atributos** criados usando engenharia de atributos.

O código separa o conjunto em atributos e rótulos, cria os subconjuntos de treinamento e teste, padroniza os atributos, treina e avalia um modelo de regressão linear.

In [ ]:
# Cria um novo DataFrame removendo a coluna MedHouseVal, ou seja, mantém apenas as variáveis de entrada (X).
# O método drop() é usado para remover a coluna 'MedHouseVal' do DataFrame.
X_nfe = df_no_feature_eng.drop(columns=['MedHouseVal'])

# Seleciona apenas a coluna MedHouseVal, que será a variável alvo (y), ou seja, aquilo que o modelo deve prever
y_nfe = df_no_feature_eng['MedHouseVal']

# Divisão em conjuntos de treinamento e teste/validação.
X_train_nfe, X_test_nfe, y_train_nfe, y_test_nfe = train_test_split(
    X_nfe, y_nfe, test_size=0.2, random_state=seed
)

# Padronização
scaler_nfe = StandardScaler()
X_train_scaled_nfe = scaler_nfe.fit_transform(X_train_nfe)
X_test_scaled_nfe = scaler_nfe.transform(X_test_nfe)

# Treina e avalia um modelo de regressão linear.
model_nfe = LinearRegression()
model_nfe.fit(X_train_nfe, y_train_nfe)

y_pred_nfe = model_nfe.predict(X_train_nfe)
print("MSE train:", mean_squared_error(y_train_nfe, y_pred_nfe))

y_pred_nfe = model_nfe.predict(X_test_nfe)
print("MSE test:", mean_squared_error(y_test_nfe, y_pred_nfe))

**16.** Compare os EQMs de treinamento e teste obtidos com os modelos treinados com e sem engenharia de atributos e responda:

+ A engenharia de atributos impactou positivamente o modelo treinado com novos atributos? (**Justifique sua resposta**).

**Resposta:**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>
